<a href="https://colab.research.google.com/github/alarcon7a/Langchain-con-Ollama/blob/main/local_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!pip install fastembed langchain langchain_community PyMuPDF chromadb
!pip install -U langchain-ollama

## Ollama desde langchain

In [2]:
# Nueva version
from langchain_ollama import OllamaLLM

# Inicializar el modelo
llm = OllamaLLM(model="llama3")

# Ejemplo de uso
response = llm.invoke("¿Qué es inteligencia artificial?")
print(response)

La inteligencia artificial (IA) se refiere a la capacidad de las máquinas o computadoras para realizar tareas que habitualmente requieren habilidades humanas, como el pensamiento, la toma de decisiones o el aprendizaje. En otras palabras, la IA es el campo de estudio y desarrollo de algoritmos y modelos matemáticos que permiten a las máquinas procesar información, aprender de ella y tomar decisiones autónomas.

La IA se basa en dos conceptos fundamentales:

1. **Procesamiento de lenguaje natural**: La capacidad de las máquinas para comprender y generar texto o voz humana, lo que les permite interactuar con humanos.
2. **Aprendizaje automático**: La capacidad de las máquinas para aprender a partir de datos y mejorar su desempeño sin necesidad de ser programadas explícitamente.

La IA se aplica en various áreas, como:

1. **Procesamiento de voz y lenguaje natural** (NLTK): reconocimiento de habla, transcripción de audio a texto, traducción automática.
2. **Visión por computadora**: análi

In [3]:
# Version antigua de Ollama
from langchain_community.llms import Ollama

llm = Ollama(model="llama3")

llm.invoke("Hola, quien eres?")

/var/folders/yn/tk2bxsk549d6hs_wgqk5z28w0000gn/T/ipykernel_66753/4086865289.py:4: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3")


'Hola! Soy LLaMA, un modelo de lenguaje artificialmente inteligente entrenado por Meta AI. Mi función es interactuar con las personas y responder a sus preguntas o mantener conversaciones sobre una variedad de temas. Estoy aquí para ayudarte con cualquier cosa que necesites o simplemente para charlar. ¿De qué quieres hablar?'

## RAG

### Cargar Documento

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("./src/HAI_2024_AI-Index-Report.pdf")

In [5]:
# cargamos las paginas del PDF
data_pdf = loader.load()

In [6]:
data_pdf[3]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'creationdate': '2024-04-23T09:25:12-07:00', 'source': './src/HAI_2024_AI-Index-Report.pdf', 'file_path': './src/HAI_2024_AI-Index-Report.pdf', 'total_pages': 502, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-23T09:32:38-07:00', 'trapped': '', 'modDate': "D:20240423093238-07'00'", 'creationDate': "D:20240423092512-07'00'", 'page': 3}, page_content='Artificial Intelligence\nIndex Report 2024\n4\nAlthough global private investment in AI decreased for the second consecutive year, investment in generative \nAI skyrocketed. More Fortune 500 earnings calls mentioned AI than ever before, and new studies show that AI \ntangibly boosts worker productivity. On the policymaking front, global mentions of AI in legislative proceedings \nhave never been higher. U.S. regulators passed more AI-related regulations in 2023 than ever before. Still, m

#### Hacemos splits del texto cada 2000 caracteres con ventana de 500

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=500)
docs = text_splitter.split_documents(data_pdf)

In [9]:
docs[2]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'creationdate': '2024-04-23T09:25:12-07:00', 'source': './src/HAI_2024_AI-Index-Report.pdf', 'file_path': './src/HAI_2024_AI-Index-Report.pdf', 'total_pages': 502, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-23T09:32:38-07:00', 'trapped': '', 'modDate': "D:20240423093238-07'00'", 'creationDate': "D:20240423092512-07'00'", 'page': 2}, page_content='Artificial Intelligence\nIndex Report 2024\n3\nMessage From  \nthe Co-directors\nA decade ago, the best AI systems in the world were unable to classify objects in images at a human level. AI \nstruggled with language comprehension and could not solve math problems. Today, AI systems routinely exceed \nhuman performance on standard benchmarks.\nProgress accelerated in 2023. New state-of-the-art systems like GPT-4, Gemini, and Claude 3 are impressively \nmultimodal: They can generate fluen

docs[3]

#### Vamos a trasnformar texto a vectores
- lo guardamos en la BD en una carpeta

In [10]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
embed_model = FastEmbedEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [11]:
from langchain_community.vectorstores import Chroma

vs = Chroma.from_documents(
    documents=docs,
    embedding=embed_model,
    persist_directory="chroma_db_dir",  # Local mode with in-memory storage only
    collection_name="stanford_report_data"
)

##### cargamos la BD persistente y cargamos 3 documentos/chunks

In [12]:
vectorstore = Chroma(embedding_function=embed_model,
                     persist_directory="chroma_db_dir",
                     collection_name="stanford_report_data")
retriever=vectorstore.as_retriever(search_kwargs={'k': 3})


/var/folders/yn/tk2bxsk549d6hs_wgqk5z28w0000gn/T/ipykernel_66753/3574404981.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(embedding_function=embed_model,


##### Creamos el prompt
- el **context** son los vectores de la BD
- **question** es la pregunta que hace el usuario

In [13]:
from langchain.prompts import PromptTemplate

custom_prompt_template = """Usa la siguiente información para responder a la pregunta del usuario.
Si no sabes la respuesta, simplemente di que no lo sabes, no intentes inventar una respuesta.

Contexto: {context}
Pregunta: {question}

Solo devuelve la respuesta útil a continuación y nada más y responde siempre en español
Respuesta útil:
"""
prompt = PromptTemplate(template=custom_prompt_template,
                        input_variables=['context', 'question'])

#### CHAINS, ahora hacemos la pregunta
- llm
- retriever, la BD de chroma

In [16]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=retriever,
                                 return_source_documents=True,
                                 chain_type_kwargs={"prompt": prompt})

In [17]:
response = qa.invoke({"query": "Cual es el comportamiento de los modelos fundacionales?"})

In [18]:
response['result']

'Según la información proporcionada, el comportamiento de los modelos fundacionales es que, desde 2019, la mayoría de ellos han originado desde la industria (72.5% en 2023). Además, la figura 1.3.19 muestra que desde 2019, Estados Unidos ha sido líder en originar la mayoría de estos modelos.'

In [22]:
response = qa.invoke({"query": "que es QLoRA?, explicamelo en detalle"})
response


{'query': 'que es QLoRA?, explicamelo en detalle',
 'result': 'No lo sé. La información proporcionada no contiene menciones a QLoRA, por lo que no puedo responder a esta pregunta.',
 'source_documents': [Document(metadata={'source': './src/HAI_2024_AI-Index-Report.pdf', 'total_pages': 502, 'modDate': "D:20240423093238-07'00'", 'trapped': '', 'page': 205, 'producer': 'Adobe PDF Library 17.0', 'keywords': '', 'creationDate': "D:20240423092512-07'00'", 'creationdate': '2024-04-23T09:25:12-07:00', 'subject': '', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'format': 'PDF 1.7', 'moddate': '2024-04-23T09:32:38-07:00', 'author': '', 'title': '', 'file_path': './src/HAI_2024_AI-Index-Report.pdf'}, page_content='206\nArtificial Intelligence\nIndex Report 2024\nChapter 3 Preview\nTable of Contents\nSlovakia’s 2023 election illustrates how AI-based \ndisinformation can be used in a political context. \nShortly before the election, a contentious audio clip \nemerged on Facebook purportedly captur

In [24]:
response['result']

'No lo sé. La información proporcionada no contiene menciones a QLoRA, por lo que no puedo responder a esta pregunta.'